[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S05/S05_07_reto_ocupacion.ipynb)

# S05_07 · Reto PBL «¿La ocupación depende de esto?» (plantilla del cuaderno entregable)
**Sprint 5 · Matemáticas y estadística en código · ⏱ 3 h (PBL)**

## Pregunta del cliente
*TurisData Canarias* nos pregunta: **«¿Cuánto cambia la ocupación diaria cuando hace más calor y cuando el día es festivo?
¿Debemos planificar personal y precios teniendo en cuenta la temperatura?»**

## Qué debes entregar (este mismo cuaderno, completo)
1. **Base de datos poblada:** un archivo `turisdata_reto.db` con la serie diaria, y los datos leídos desde Python con SQL.
2. **Regresión lineal desde cero** con descenso del gradiente (simple y con controles), sin `scikit-learn` en la implementación.
3. **Contraste con una librería** (`statsmodels`) y comparación de coeficientes.
4. **Análisis estadístico** con contrastes e intervalos de confianza.
5. **Respuesta al cliente** (máx. 150 palabras) con recomendación, límites de lo que se puede afirmar y una nota de IA responsable.

Los **puntos de control** (`assert`) te avisan si algo no cuadra; el análisis y la respuesta se evalúan con la rúbrica del curso
(corrección técnica, reproducibilidad, análisis y comunicación al cliente, IA responsable).

> Las celdas con `...` / `# TODO` son tuyas. Puedes reutilizar el código de los cuadernos S05_03 y S05_05.
> Los datos son sintéticos: las conclusiones son para practicar, no estadísticas reales.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import os
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats

SEMILLA = 42
RUTA_BD = "turisdata_reto.db"        # el archivo que entregarás junto al cuaderno

## 1 · Base de datos: cargar y consultar
Crea la base `turisdata_reto.db` con una tabla `ocupacion_diaria` (a partir de `ocupacion_diaria.csv`) y un índice sobre
`fecha`. Después **lee los datos desde la base con SQL** en un DataFrame `diaria` que incluya una columna `mes` (entero 1-12).

In [ ]:
# TODO: to_sql con if_exists="replace"; CREATE INDEX IF NOT EXISTS ... ON ocupacion_diaria(fecha); SELECT ..., CAST(SUBSTR(fecha, 6, 2) AS INTEGER) AS mes
...
assert os.path.exists(RUTA_BD), "Debe existir el archivo turisdata_reto.db"
assert len(diaria) == 1096, f"Deben leerse 1096 días y has leído {len(diaria)}"
assert "mes" in diaria.columns and diaria["mes"].between(1, 12).all(), "Falta la columna mes (1-12)"
assert con.execute("SELECT COUNT(*) FROM sqlite_master WHERE type='index' AND name='idx_ocupacion_fecha'").fetchone()[0] == 1, "Falta el índice idx_ocupacion_fecha"
print("Punto de control 1 superado")

## 2 · Primer vistazo a los datos
Describe la serie (media, dispersión, meses de mayor y menor ocupación) y dibuja **al menos dos gráficos** con títulos y ejes
en español (por ejemplo, ocupación por mes y ocupación frente a temperatura). Escribe debajo 2–3 observaciones.

In [ ]:
# TODO: describe(), una media por mes con groupby y dos gráficos (boxplot por mes y dispersión temperatura-ocupación)
...


**Mis observaciones (escribe aquí):**
- ...

## 3 · Regresión lineal desde cero (descenso del gradiente)
Implementa `gradiente_mse(X, y, w)` y `entrenar(X, y, tasa, epocas)` (puedes basarte en S05_03; **no uses scikit-learn aquí**).
Ajusta dos modelos:
- **Modelo simple:** `ocupacion ~ temperatura` (variable estandarizada; deshaz la estandarización para dar la pendiente en puntos por °C).
- **Modelo con control:** `ocupacion ~ temperatura + festivo + indicadores de mes` (11 columnas de meses; estandariza todas las variables;
  usa unas 2.000 épocas con tasa 0,1).

Guarda `pendiente_simple` (puntos por °C), `efecto_temp_control` y `efecto_festivo_control` (ambos en puntos, ya en unidades originales).

In [ ]:
# TODO: gradiente (2/n) X^T (Xw - y); bucle de descenso; en el modelo con control, get_dummies(mes, drop_first=True), estandariza y divide cada coeficiente entre la desviación de su variable
...
assert abs(pendiente_simple + 1.86) < 0.02, f"La pendiente simple debería ser ≈ -1.86 puntos/°C y tienes {pendiente_simple:.3f}"
assert abs(efecto_festivo_control - 9.11) < 0.1, f"El efecto del festivo con control debería ser ≈ +9.11 y tienes {efecto_festivo_control:.2f}"
assert abs(efecto_temp_control + 0.09) < 0.05, f"El efecto de la temperatura con control debería ser ≈ -0.09 y tienes {efecto_temp_control:.3f}"
print(f"Punto de control 2 superado: simple {pendiente_simple:.2f} · con control: temp {efecto_temp_control:.2f}, festivo {efecto_festivo_control:+.2f}")

Dibuja la **curva de aprendizaje** (MSE por época) de uno de los modelos y comenta en una línea si ha convergido.

In [ ]:
# TODO: plt.plot(historial) con títulos y ejes en español
...


## 4 · Contraste con una librería
Ajusta con `statsmodels` (`smf.ols`) los mismos dos modelos. Guarda `ols_simple` y `ols_control` y crea una tabla
`comparacion` (DataFrame) con tres filas (`pendiente_simple`, `efecto_temp_control`, `efecto_festivo_control`) y dos columnas:
`desde_cero` y `statsmodels`.

In [ ]:
# TODO: smf.ols("ocupacion_pct ~ temperatura_c", ...) y "ocupacion_pct ~ temperatura_c + festivo + C(mes)"; params["..."]
...
assert comparacion.shape == (3, 2), "La tabla debe tener 3 filas y 2 columnas"
assert np.allclose(comparacion["desde_cero"], comparacion["statsmodels"], atol=0.02), f"Tu implementación no coincide con statsmodels:\n{comparacion}"
print("Punto de control 3 superado")
print(comparacion.round(3))

## 5 · Responder con estadística
Con `ols_control` y con un contraste, calcula y guarda:
- `p_temp_control`: p-valor del efecto de la temperatura **con control**.
- `ic_festivo`: intervalo de confianza del 95 % del efecto del festivo (tupla de dos números, con `conf_int`).
- `p_festivo_ttest`: p-valor del t de Welch entre ocupación de festivos y no festivos.
- `r2_control`: R² del modelo con control.

In [ ]:
# TODO: ols_control.pvalues[...], ols_control.conf_int().loc["festivo"], stats.ttest_ind(..., equal_var=False).pvalue, ols_control.rsquared
...
assert p_temp_control > 0.05, f"Con control de mes, la temperatura no es significativa (p ≈ 0.48) y tienes p = {p_temp_control:.3f}"
assert abs(ic_festivo[0] - 7.35) < 0.05 and abs(ic_festivo[1] - 10.87) < 0.05, f"El IC del festivo debería ser ≈ (7.35, 10.87) y tienes {ic_festivo}"
assert p_festivo_ttest < 1e-20, "El t-test de festivos debe dar un p-valor minúsculo"
assert 0.55 < r2_control < 0.62, f"El R² del modelo con control ≈ 0.58 y tienes {r2_control:.3f}"
print("Punto de control 4 superado")

## 6 · Respuesta al cliente (máx. 150 palabras)
Escribe en la celda de abajo, **para una persona no técnica**:
1. **Respuesta directa** a las dos preguntas (¿la temperatura? ¿los festivos?) con las cifras clave y su incertidumbre.
2. **Recomendación accionable** (personal, precios).
3. **Límites:** ¿qué NO puedes afirmar? (correlación frente a causalidad, variable de confusión "mes", datos sintéticos de un solo alojamiento, R²).
4. **IA responsable:** una frase sobre riesgos o cautelas (por ejemplo, usar el modelo para fijar precios sin revisión humana, o presentarlo como causal).

> **Mi respuesta al cliente:**
>
> ...

## 7 · Reproducibilidad: antes de entregar
- [ ] *Reiniciar y ejecutar todo* funciona sin errores.
- [ ] Se entrega `turisdata_reto.db` (o el cuaderno lo crea al ejecutarse) y las versiones de las librerías están indicadas en el README (o en una celda `!pip freeze`).
- [ ] Todas las gráficas tienen título y ejes en español.
- [ ] La respuesta al cliente cabe en 150 palabras.

In [ ]:
con.close()